# Titanic survival — tree model code

Data cleaning and feature engineering, preprocessing, tree training and evaluation. The split precedes missing-`Embarked` removal, unlike the main notebook.

Historical feature choices were informed by full-dataset EDA. The evaluation subset was reused for model comparisons; age imputation, ticket frequencies and encoding occur before CV. Results remain exploratory, with no independent final evaluation. [Project Documentation](Project%20Documentation/README.md).

**Code assistance:** The code was developed with assistance from **CODEX**.


## 1. Load


In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier


df = pd.read_csv("Data/Titanic_dataset.csv")


## 2. Clean and derive features


In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")


## 3. Split


In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


## 4. Preprocess


In [ ]:
# Exclude the passenger identifier.
X_train = X_train.drop(columns=["PassengerId"])
X_test = X_test.drop(columns=["PassengerId"])

# Normalize titles and exclude the original name.
title_mapping = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}

X_train["Title"] = X_train["Title"].replace(title_mapping)
X_test["Title"] = X_test["Title"].replace(title_mapping)

common_titles = ["Mr", "Mrs", "Miss", "Master"]

X_train["Title"] = X_train["Title"].where(
    X_train["Title"].isin(common_titles),
    "Rare"
)

X_test["Title"] = X_test["Title"].where(
    X_test["Title"].isin(common_titles),
    "Rare"
)

X_train = X_train.drop(columns=["Name"])
X_test = X_test.drop(columns=["Name"])

# Impute age using the training median.
age_median = X_train["Age"].median()

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

# Derive ticket-group size from training frequencies.
ticket_group_size = X_train["Ticket"].value_counts()

X_train["TicketGroupSize"] = (
    X_train["Ticket"]
    .map(ticket_group_size)
    .fillna(1)
)

X_test["TicketGroupSize"] = (
    X_test["Ticket"]
    .map(ticket_group_size)
    .fillna(1)
)

X_train = X_train.drop(columns=["Ticket"])
X_test = X_test.drop(columns=["Ticket"])

# Exclude cabin.
X_train = X_train.drop(columns=["Cabin"])
X_test = X_test.drop(columns=["Cabin"])

# Remove missing departure ports after the split.
train_mask = X_train["Embarked"].notna()

X_train = X_train.loc[train_mask]
y_train = y_train.loc[train_mask]

test_mask = X_test["Embarked"].notna()

X_test = X_test.loc[test_mask]
y_test = y_test.loc[test_mask]

# Encode categories separately and align test columns to training.
categorical_columns = ["Sex", "Embarked", "Title"]

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)


## 5. Train and select models


In [ ]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train, y_train)

rf_grid_model = RandomForestClassifier(
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6, 7],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    estimator=rf_grid_model,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

dt_grid_model = DecisionTreeClassifier(
    random_state=42
)

param_grid_dt = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 10],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["gini", "entropy"]
}

grid_search_dt = GridSearchCV(
    estimator=dt_grid_model,
    param_grid=param_grid_dt,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search_dt.fit(X_train, y_train)

print("Selected Random Forest parameters:", grid_search_rf.best_params_)
print(f"Random Forest cross-validation F1: {grid_search_rf.best_score_:.4f}")
print("Selected Decision Tree parameters:", grid_search_dt.best_params_)
print(f"Decision Tree cross-validation F1: {grid_search_dt.best_score_:.4f}")


## 6. Evaluate


In [ ]:
def evaluate_model(name, estimator):
    predictions = estimator.predict(X_test)
    probabilities = estimator.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities),
    }


results = pd.DataFrame([
    evaluate_model("Decision Tree baseline", decision_tree),
    evaluate_model("Random Forest baseline", random_forest),
    evaluate_model("Random Forest selected by CV", grid_search_rf.best_estimator_),
    evaluate_model("Decision Tree selected by CV", grid_search_dt.best_estimator_),
]).set_index("Model")
results.round(4)
